In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# =============================================================================
# Custom Expanding Window Encoder Transformer
# =============================================================================
class ExpandingWindowEncoder(BaseEstimator, TransformerMixin):
    """
    This transformer computes an expanding window average of the target ('score')
    for each group defined by a categorical column (e.g. 'subject') using only
    data from earlier dates than the current observation.

    For training data (in fit_transform), each row's new feature is computed by
    taking the average of all previous target values for that subject.

    For new data (in transform), it uses the training data stored during fit:
    for each row, it computes the average of training target values for that subject
    with dates strictly before the row's date.
    """

    def __init__(self, categorical_column, date_column):
        self.categorical_column = categorical_column  # e.g., 'subject'
        self.date_column = date_column               # e.g., 'date'

    def fit(self, X, y):
        # Ensure we work on a copy and that the date column is datetime.
        X_ = X.copy()
        X_[self.date_column] = pd.to_datetime(X_[self.date_column])
        X_['_target'] = y

        # Compute a global mean as a fallback.
        self.global_mean_ = y.mean()

        # For use in transform: store, per category, the training data (date and target)
        self.train_data_ = {}
        for subj, group in X_.groupby(self.categorical_column):
            group_sorted = group.sort_values(self.date_column)
            self.train_data_[subj] = group_sorted[[self.date_column, '_target']]
        return self

    def transform(self, X):
        # For new (or test) data, compute the expanding average based solely on training data.
        X_ = X.copy()
        X_[self.date_column] = pd.to_datetime(X_[self.date_column])
        new_feature = []
        for _, row in X_.iterrows():
            subj = row[self.categorical_column]
            current_date = row[self.date_column]
            if subj not in self.train_data_:
                # If the subject wasn’t seen in training, use the global mean.
                new_feature.append(self.global_mean_)
            else:
                train_group = self.train_data_[subj]
                # Select only training rows with a date earlier than the current date.
                valid = train_group[train_group[self.date_column] < current_date]
                if valid.empty:
                    new_feature.append(self.global_mean_)
                else:
                    new_feature.append(valid['_target'].mean())
        X_[self.categorical_column + '_expanding'] = new_feature
        return X_

    def fit_transform(self, X, y):
        # When fitting on training data, we can compute the expanding average per subject.
        self.fit(X, y)
        X_ = X.copy()
        X_[self.date_column] = pd.to_datetime(X_[self.date_column])
        X_['_target'] = y
        new_feature = pd.Series(index=X_.index, dtype=float)

        # Process each subject group separately.
        for subj, group in X_.groupby(self.categorical_column):
            group_sorted = group.sort_values(self.date_column)
            # Compute cumulative sum and count for the target, then shift by one so that the
            # current row is excluded (i.e., only prior observations are used).
            cumsum = group_sorted['_target'].cumsum().shift(1)
            count = np.arange(len(group_sorted))  # count: 0, 1, 2, ...
            avg = cumsum / count
            # For the first row in each group (where count is 0), fall back to the global mean.
            avg[count == 0] = self.global_mean_
            new_feature.loc[group_sorted.index] = avg
        X_[self.categorical_column + '_expanding'] = new_feature
        X_ = X_.drop(columns=['_target'])
        return X_

# =============================================================================
# Sample Data Creation
# =============================================================================
# Example dataset with a date column.
data = {
    'subject': ['Math', 'Science', 'Math', 'English', 'Science', 'Math', 'English', 'Science'],
    'feature1': [1, 2, 3, 4, 5, 6, 7, 8],
    'score': [80, 85, 78, 90, 88, 75, 92, 87],
    'date': ['2020-01-01', '2020-01-02', '2020-01-03', '2020-01-04',
             '2020-01-05', '2020-01-06', '2020-01-07', '2020-01-08']
}
df = pd.DataFrame(data)
df['date'] = pd.to_datetime(df['date'])

# Separate features and target.
X = df[['subject', 'feature1', 'date']]
y = df['score']

# Split into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# =============================================================================
# Build the Pipeline Using the ExpandingWindowEncoder
# =============================================================================
pipeline_rolling = Pipeline([
    # Step 1: Apply our custom feature engineering transformer.
    ('feature_engineering', ExpandingWindowEncoder(categorical_column='subject', date_column='date')),

    # Step 2: Use ColumnTransformer to select the features for the regression.
    ('column_selector', ColumnTransformer([
        ('num_features', 'passthrough', ['feature1']),
        ('expanding_feature', 'passthrough', ['subject_expanding'])
    ])),

    # Step 3: Fit a linear regression model.
    ('regressor', LinearRegression())
])

# =============================================================================
# Fit the Pipeline and Make Predictions
# =============================================================================
pipeline_rolling.fit(X_train, y_train)
y_pred_rolling = pipeline_rolling.predict(X_test)
print("Predictions with Expanding Window Encoder on test set:", y_pred_rolling)
